# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, their fields/columns, and their `@id`s.

`mlcroissant` represents each RecordSet using a unique `@id`. We'll list all record sets and, for each, display available fields/columns by `@id` and name.

In [ ]:
# List all record sets and their fields/columns
if hasattr(metadata, 'record_sets'):
    record_sets = metadata.record_sets
elif hasattr(metadata, 'recordSet'):
    # Also check alternate spelling used in Croissant
    record_sets = metadata.recordSet
else:
    record_sets = []

print("Found record sets:")
record_set_ids = []
for rs in record_sets:
    print(f"  - {rs['@id']} : {rs['name'] if 'name' in rs else '(no name)'}")
    record_set_ids.append(rs["@id"])
    # List fields/columns for each record set
    if 'fields' in rs and rs['fields']:
        print("    Fields:")
        for field in rs['fields']:
            print(f"      • {field['@id']}: {field.get('name', '')}")
    if 'columns' in rs and rs['columns']:
        print("    Columns:")
        for col in rs['columns']:
            print(f"      • {col['@id']}: {col.get('name', '')}")
if not record_sets:
    print("No record sets found or Croissant schema does not directly list them.")

## 3. Data Extraction
Load data from the main record set into a DataFrame for analysis. We'll use the record set and field/column `@id`s identified above.

> **NOTE:** If your dataset does not print any record sets above, you may need to view the Croissant schema manually or refer to the dataset's documentation to identify an appropriate record set `@id`.

Below, we look for a main record set and extract data from it.

In [ ]:
# Collect all record set IDs from previous cell
if len(record_set_ids):
    print("Record set IDs:", record_set_ids)
else:
    # Fallback: Try dataset.records() directly to inspect top-level records
    print("No record_set_ids found, attempting default loading...")
    record_set_ids = []

# Try to load data from each record set
dataframes = {}
for rsid in record_set_ids:
    # mlcroissant expects the record set @id as argument
    try:
        records = list(dataset.records(record_set=rsid))
        dataframes[rsid] = pd.DataFrame(records)
        print(f"Loaded {len(records)} records from {rsid}")
    except Exception as e:
        print(f"Could not load records for {rsid}: {e}")

if dataframes:
    # Pick the first available record set
    main_id = list(dataframes.keys())[0]
    print(f"Using record set: {main_id}")
    print("Columns:")
    print(dataframes[main_id].columns.tolist())
    dataframes[main_id].head()
else:
    main_id = None
    print("No dataframes loaded.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. Example operations include:
- Removing outliers
- Transforming data distributions
- Grouping data by attributes

> **Note:** All references to record set or field/column use the entity's `@id`.

In [ ]:
# Example: Filter and normalize a numeric field
# Edit the following IDs as per your schema output above
import numpy as np

if main_id is not None:
    df = dataframes[main_id]
    # Attempt to auto-select a numeric column
    numerics = df.select_dtypes(include=[np.number]).columns.tolist()
    if numerics:
        numeric_field = numerics[0]
        print(f"Using numeric field for EDA: {numeric_field}")
        threshold = df[numeric_field].mean() if not np.isnan(df[numeric_field].mean()) else 0
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        print(filtered_df.head())

        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Attempt to group by another column
        group_candidates = [c for c in df.columns if c != numeric_field]
        group_field = group_candidates[0] if group_candidates else None
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"Grouped mean {numeric_field} by {group_field}:")
            print(grouped_df.head())
    else:
        print("No numeric columns found for EDA.")
else:
    print("No data to analyze.")

## 5. Visualization
Visualize distributions or relationships between fields in the dataset. Below is an example using Matplotlib and/or Seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_id is not None and numerics:
    numeric_field = numerics[0]
    plt.figure(figsize=(8,5))
    sns.histplot(df[numeric_field], kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.show()

    # Example pairplot with up to two numerics if available
    if len(numerics) > 1:
        sns.pairplot(df[numerics[:2]].dropna())
        plt.show()
else:
    print("No numeric fields for visualization or no main data.")

## 6. Conclusion
In this notebook, we loaded the FAIR² dataset package using its Croissant schema, identified the record sets and their fields, extracted and explored records from a main record set, and performed basic EDA including filtering and normalization as well as plotting key distributions.

- For robust analysis, study the full Croissant schema to select meaningful fields and groupings.
- All references to record set, field, and column entities are via their `@id`, in keeping with the FAIR principles and Croissant's recommendations.

Further investigation can include integrating external protein databases, advanced statistical analysis, or machine learning pipelines tailored to your research.